####Introduction to Aggregation
Aggregation in Spark are implemented using functions

#####Comonly used aggregate functions
1. count(*), count(expr), count(DISTINCT expr)
2. min(expr), max(expr), avg(expr), sum(expr)

#####Types of Aggregation
1. Simple Aggregation - Returns only one single value like avag and sum
2. Grouped Aggregation
3. Multilevel Aggregation
4. Window Aggregation



####Requirement - Analysis Data Set

Prepare club bookings dataset for analysis
```
+----------+------------+---------------+-------------------+--------------+
|booking_id| member_name|  facility_name|         start_time|booking_amount|
+----------+------------+---------------+-------------------+--------------+
```

In [0]:
bookings_df = spark.table("dev.spark_db.bookings")
facilities_df = spark.table("dev.spark_db.facilities")
members_df = spark.table("dev.spark_db.members")

club_bookings_df = (
    bookings_df.join(facilities_df, "facility_id")
            .join(members_df, "member_id", "left")
            .selectExpr("booking_id",
                        "case when member_id==0 then 'Guest Member' else concat_ws(' ', first_name, last_name) end as member_name",
                        "facility_name","start_time",
                        "case when member_id == 0 then slots * guest_cost else slots * member_cost end as booking_amount")            
)

club_bookings_df.display()

####1. Calculate total earnings and average booking value.

1.1 Using sql like expressions

In [0]:
result_df = (
    club_bookings_df.selectExpr(
        "sum(booking_amount) as total_earning",
        "avg(booking_amount) as avg_booking_amount"
    )
)

result_df.display()

1.2 Using column expressions

In [0]:
from pyspark.sql.functions import sum,avg

result_df =(
    club_bookings_df.select(
        sum("booking_amount").alias("total_earning"),
        avg("booking_amount").alias("avg_booking_amount")
    )
)

result_df.display()

1.3 Using aggregate transformation

In [0]:
from pyspark.sql.functions import sum, avg
# Use agg rather than select and selectExpr for aggregation because it is more efficient and works for complex aggregations.
result_df = (
    club_bookings_df.agg(sum("booking_amount").alias("total_earning"),
                         avg("booking_amount").alias("avg_booking_value"))
)

result_df.display()